# DeepVF benchmark

In [8]:
# library
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch

import os
import src

tqdm.pandas()

# directory
root = Path.cwd()
base_dir = os.path.join(root, "data", "DeepVF")
os.chdir(base_dir)

In [12]:
def preprocess(fasta_path, desc_path, tax_path, id_field="id", replace=True):
    df = src.fasta_to_df(fasta_path)

    # interproscan
    scan_df = pd.read_csv(desc_path)
    df = df.merge(scan_df)

    # taxonomy
    tax_df = pd.read_csv(tax_path)
    df = df.merge(tax_df)

    if replace:
        mask = df["sequence"].str.contains("J|\\*", regex=True, na=False)
        print("rows with J or *:", mask.sum())
        df.loc[mask, "sequence"] = df.loc[mask, "sequence"].str.replace("J|\\*", "X", regex=True)
    
    df = df.drop(columns=["taxid"]) # drop index
    df = df.drop_duplicates(subset=["id"], keep="first").reset_index(drop=True)

    return df

train_pos = preprocess("DeepVF_Training_Dataset/train_positive.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
train_neg = preprocess("DeepVF_Training_Dataset/train_negative.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
test_pos  = preprocess("DeepVF_Independent_Dataset/ind_positive.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")
test_neg  = preprocess("DeepVF_Independent_Dataset/ind_negative.fasta", "df_interproscan_no_dup_semantic.csv", "df_taxonomy_gtdb.csv")

train_pos["label"] = 1.0
test_pos["label"]  = 1.0
train_neg["label"] = 0.0
test_neg["label"]  = 0.0

rows with J or *: 0
rows with J or *: 0
rows with J or *: 0
rows with J or *: 0


In [13]:
train = pd.concat([train_pos, train_neg])
test  = pd.concat([test_pos, test_neg])

In [25]:
y_train = list(train['label'])
y_test = list(test['label'])

In [16]:
X_train_esm = src.esm_extract(train)
X_test_esm = src.esm_extract(test)

Using cache found in /home/sjchoi/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 7334/7334 [06:33<00:00, 18.62it/s] 
Using cache found in /home/sjchoi/.cache/torch/hub/facebookresearch_esm_main
100%|██████████| 1152/1152 [01:01<00:00, 18.69it/s]


In [17]:
from transformers import AutoTokenizer, AutoModel

def bert_extract(df):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained("google-bert/bert-base-uncased")
    model = AutoModel.from_pretrained("google-bert/bert-base-uncased").to(device)
    model.eval()
    txt_emb = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"Extracting"):
        desc = str(row.get('desc_nodup', '')).strip()
        name = str(row.get('name', '')).strip()
        rank = str(row.get('rank', '')).strip()
        combined_text = f"{desc}|{name}|{rank}"
        embedding = src.get_embedding(combined_text, tokenizer, model, device)
        txt_emb.append(embedding.numpy().tolist())
    
    return np.array(txt_emb)

In [18]:
X_train_text = bert_extract(train)
X_test_text  = bert_extract(test)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 898.28it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1148.19it/s, Materializing param=pooler.dense.weight]              

In [ ]:
# np.save("embedding/bert_train.npy", X_train_text)
# np.save("embedding/bert_test.npy", X_test_text)

In [23]:
X_train_all = np.concatenate([X_train_esm, X_train_text], axis=1)
X_test_all  = np.concatenate([X_test_esm, X_test_text], axis=1)

In [24]:
X_train_all.shape

(7334, 2048)

In [29]:
import warnings
warnings.filterwarnings(action='ignore', category=FutureWarning, module='xgboost')

from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, confusion_matrix

# Define the parameter grid to search
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0]
}

# Initialize XGBClassifier
xgb = XGBClassifier(
    tree_method="hist",
    eval_metric="auc",
    random_state=42,
)

# Grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=5, scoring='roc_auc', verbose=1, n_jobs=-1, error_score="raise")

# Fit grid search
grid_search.fit(X_train_all, y_train)

print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation AUC: ", grid_search.best_score_)

# Evaluate on test data using best estimator
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

Fitting 5 folds for each of 108 candidates, totalling 540 fits


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}
Best cross-validation AUC:  0.8832061380726725


Best parameters found:  {'colsample_bytree': 0.7, 'learning_rate': 0.2, 'max_depth': 7, 'n_estimators': 200, 'subsample': 1.0}

In [30]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9381721402391976
Classification Report:
               precision    recall  f1-score   support

         0.0       0.82      0.94      0.88       576
         1.0       0.93      0.80      0.86       576

    accuracy                           0.87      1152
   macro avg       0.88      0.87      0.87      1152
weighted avg       0.88      0.87      0.87      1152

Accuracy: 0.870
F1-score: 0.859
Precision: 0.935
Recall: 0.795
MCC: 0.748


In [38]:
best_params = {
    'colsample_bytree': 0.7,
    'learning_rate': 0.2,
    'max_depth': 7,
    'n_estimators': 200,
    'subsample': 1.0,
}

from xgboost import XGBClassifier

best_model = XGBClassifier(
    **best_params,
    eval_metric="auc",
    tree_method="hist",
    random_state=42,
)

best_model.fit(X_train_all, y_train)
y_pred = best_model.predict(X_test_all)
y_proba = best_model.predict_proba(X_test_all)[:, 1]

print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")
tn, fp, fn, tp = (confusion_matrix(y_test, y_pred)).ravel().tolist()
print(f"specificity: {tn/(tn+fp):.3f}")
print(f"Precision: {precision:.3f}")

Test ROC AUC: 0.9381721402391976
Classification Report:
               precision    recall  f1-score   support

         0.0       0.82      0.94      0.88       576
         1.0       0.93      0.80      0.86       576

    accuracy                           0.87      1152
   macro avg       0.88      0.87      0.87      1152
weighted avg       0.88      0.87      0.87      1152

Accuracy: 0.870
F1-score: 0.859
Recall: 0.795
MCC: 0.748
specificity: 0.944
Precision: 0.935


In [28]:
print("Test ROC AUC:", roc_auc_score(y_test, y_proba))
print("Classification Report:\n", classification_report(y_test, y_pred))

accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
mcc = matthews_corrcoef(y_test, y_pred)

print(f"Accuracy: {accuracy:.3f}")
print(f"F1-score: {f1:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"MCC: {mcc:.3f}")

Test ROC AUC: 0.9381721402391976
Classification Report:
               precision    recall  f1-score   support

         0.0       0.82      0.94      0.88       576
         1.0       0.93      0.80      0.86       576

    accuracy                           0.87      1152
   macro avg       0.88      0.87      0.87      1152
weighted avg       0.88      0.87      0.87      1152

Accuracy: 0.870
F1-score: 0.859
Precision: 0.935
Recall: 0.795
MCC: 0.748


In [31]:
import joblib

# After training and grid search as before
best_model = grid_search.best_estimator_

# Save the model to a file
joblib.dump(best_model, 'DeepVF_xgb_model.pkl')



['DeepVF_xgb_model.pkl']

In [36]:
import pandas as pd
import joblib

# Load the trained model
loaded_model = joblib.load('DeepVF_xgb_model.pkl')

# Load your test set (replace 'test.csv' with your actual test file path)

labels = test['label']

# Predict
probabilities = loaded_model.predict_proba(X_test_all)[:,1]
predictions = loaded_model.predict(X_test_all)


# Prepare result dataframe with id, predicted label, and original label
result_df = pd.DataFrame({
    'id': test['id'],
    'prob': probabilities,
    'pred': predictions,
    'label': labels
})

# Save results
result_df.to_csv('VFTextSeq_test_prediction.csv', index=False)
